In [0]:
import random
import uuid
from datetime import datetime, timedelta
import builtins
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit
import re

round = builtins.round

start_datetime = datetime(2025, 5, 6, 9, 0)

# Añade columna 'timestamp' con valor fijo
def add_fixed_timestamp_column(df: DataFrame, partition_dt: datetime) -> DataFrame:
    return df.withColumn("timestamp", lit(partition_dt.isoformat()))

# Devuelve el último timestamp encontrado en los nombres de archivo (ya que no hay carpetas)
def get_next_partition_datetime(base_path: str) -> datetime:
    pattern = re.compile(r"(\d{4})-(\d{2})-(\d{2})-(\d{2})-(\d{2})")
    timestamps = []

    try:
        items = dbutils.fs.ls(base_path)
        for item in items:
            match = pattern.search(item.name)
            if match:
                timestamps.append(datetime(
                    int(match.group(1)),
                    int(match.group(2)),
                    int(match.group(3)),
                    int(match.group(4)),
                    int(match.group(5))
                ))
    except:
        pass

    if timestamps:
        last = max(timestamps)
        return last + timedelta(hours=1)
    else:
        return start_datetime

# Escribe el archivo en base_path con timestamp en el nombre y sin subcarpetas
def write_partitioned(df: DataFrame, base_path: str, partition_dt: datetime, format: str = "csv", header: bool = False, coalesce: bool = False, dataset_name: str = ""):
    df = add_fixed_timestamp_column(df, partition_dt)

    timestamp_str = partition_dt.strftime("%Y-%m-%d-%H-%M")
    temp_path = f"{base_path.rstrip('/')}/temp/"

    writer = df.coalesce(1).write if coalesce else df.write
    writer = writer.mode("overwrite")
    if format == "csv":
        writer.option("header", header).csv(temp_path)
    elif format == "json":
        writer.json(temp_path)
    else:
        raise ValueError("Formato no soportado: usa 'csv' o 'json'")

    files = dbutils.fs.ls(temp_path)
    part_file = next((f.path for f in files if f.name.startswith("part-") and f.name.endswith(f".{format}")), None)

    if part_file:
        final_name = f"{dataset_name}_{timestamp_str}.{format}"
        final_path = f"{base_path.rstrip('/')}/{final_name}"
        dbutils.fs.mv(part_file, final_path)
        dbutils.fs.rm(temp_path, recurse=True)
        print(f"✔ Archivo generado: {final_path}")
    else:
        print("❌ No se encontró archivo 'part-*' para renombrar.")

# Rutas base
BASE_PATHS = {
    "ventas": "abfss://landing@mastervdrp001sa.dfs.core.windows.net/retail/sales_orders/",
    "inventario": "abfss://landing@mastervdrp001sa.dfs.core.windows.net/inventory/inventario/",
    "logistica": "abfss://landing@mastervdrp001sa.dfs.core.windows.net/logistics/shipments/",
    "meteorologia": "abfss://landing@mastervdrp001sa.dfs.core.windows.net/weather/forecast_data/"
}

# Repetimos 10 veces por dataset
for i in range(10):
    # Timestamp para esta iteración
    base_timestamp = get_next_partition_datetime(BASE_PATHS["ventas"]) + timedelta(hours=i)

    # Ventas
    ventas = [
        {
            "order_id": str(uuid.uuid4()),
            "customer_id": random.randint(1, 1000),
            "product_id": f"SKU{random.randint(1000,9999)}",
            "product_category": random.choice(["fruta", "hortaliza", "semillas", "fertilizante"]),
            "quantity": random.randint(1, 20),
            "unit_price": round(random.uniform(1.5, 150.0), 2),
            "currency": "EUR",
            "payment_method": random.choice(["tarjeta", "paypal", "transferencia"]),
            "shipping_cost": round(random.uniform(0, 10), 2),
            "order_date": base_timestamp.isoformat()
        }
        for _ in range(100)
    ]
    df_ventas = spark.createDataFrame(ventas)
    write_partitioned(df_ventas, BASE_PATHS["ventas"], partition_dt=base_timestamp, format="json", dataset_name="ventas")

    # Inventario
    inventario = [
        (
            f"SKU{random.randint(1000,9999)}",
            random.randint(0, 1000),
            random.choice(["almacén_A", "almacén_B"]),
            random.choice(["fruta", "hortaliza", "semillas", "fertilizante"]),
            round(random.uniform(0.1, 25.0), 2),
            base_timestamp.isoformat()
        )
        for _ in range(100)
    ]
    df_inventario = spark.createDataFrame(inventario, ["sku", "stock_actual", "ubicacion", "categoria", "peso_kg", "ultima_actualizacion"])
    write_partitioned(df_inventario, BASE_PATHS["inventario"], partition_dt=base_timestamp, format="csv", header=True, coalesce=True, dataset_name="inventario")

    # Logística
    logistica = [
        (
            f"proveedor_{j}",
            random.choice(["transporte_maritimo", "terrestre", "aereo"]),
            random.randint(1, 10),
            random.choice(["en camino", "entregado", "retrasado"]),
            (base_timestamp - timedelta(days=random.randint(1, 15))).isoformat(),
            (base_timestamp + timedelta(days=random.randint(1, 10))).isoformat()
        )
        for j in range(100)
    ]
    df_logistica = spark.createDataFrame(logistica, ["proveedor", "modo_transporte", "num_envios", "estado_envio", "fecha_envio", "fecha_estimacion_entrega"])
    write_partitioned(df_logistica, BASE_PATHS["logistica"], partition_dt=base_timestamp, format="csv", header=True, coalesce=True, dataset_name="logistica")

    # Meteorología
    meteorologia = [
        {
            "region": f"zona_{random.randint(1, 5)}",
            "temperatura": round(random.uniform(10.0, 35.0), 1),
            "humedad": random.randint(30, 90),
            "precipitacion_mm": round(random.uniform(0.0, 10.0), 2),
            "velocidad_viento_kmh": round(random.uniform(0.0, 50.0), 2),
            "fecha": (base_timestamp - timedelta(days=j)).isoformat()
        }
        for j in range(100)
    ]
    df_meteo = spark.createDataFrame(meteorologia)
    write_partitioned(df_meteo, BASE_PATHS["meteorologia"], partition_dt=base_timestamp, format="json", dataset_name="meteorologia")


✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/retail/sales_orders/ventas_2025-05-06-09-00.json
✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/inventory/inventario/inventario_2025-05-06-09-00.csv
✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/logistics/shipments/logistica_2025-05-06-09-00.csv
✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/weather/forecast_data/meteorologia_2025-05-06-09-00.json
✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/retail/sales_orders/ventas_2025-05-06-11-00.json
✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/inventory/inventario/inventario_2025-05-06-11-00.csv
✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/logistics/shipments/logistica_2025-05-06-11-00.csv
✔ Archivo generado: abfss://landing@mastervdrp001sa.dfs.core.windows.net/weather/forecast_data/meteorologia_2025-05-06-11-00.j